# 第 44 课：自监督语音预训练——不给转录也能学声学表示

人工转录昂贵，但原始音频极多。自监督学习（SSL）先用音频本身构造训练目标，学到可迁移的声学表示，再用较少转录数据微调 ASR。

本课用“遮住若干声学帧、根据上下文恢复离散单元”的 CPU 小实验贯通 wav2vec 2.0 / HuBERT 类方法的核心逻辑。

## 完成标准

1. 区分自监督预训练和有标签 ASR 微调；
2. 解释 feature encoder、mask、context encoder、target/codebook；
3. 说明为什么 loss 只能在被 mask 的位置计算；
4. 亲手训练一个 masked acoustic model，使遮挡位置准确率显著高于随机猜测；
5. 把预训练 encoder 接到 CTC head，验证梯度链路。

重要边界：本课使用合成“声学单元”，用于验证机制，不代表真实语音效果。

## 课前诊断

1. 没有文字转录时，神经网络还能从音频中获得什么监督信号？
2. 如果目标帧没有被遮住，模型可能采用什么投机办法？
3. 为什么预训练好以后仍需要少量带文字的数据？

<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：CTC/流式状态和延迟；训练/测试数据权限；基线、消融与外部评测证据。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](43_RNNT与TDT_二维对齐和跳帧.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：明确任务、数据、算力、延迟与风险约束
  ↓ 本课要学会的变换、状态或判断
输出：能与 CTC/RNN-T/AED/LALM 基线公平比较的现代模型实验
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [1]:
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(21)
np.random.seed(21)
random.seed(21)
torch.set_num_threads(2)
device = torch.device("cpu")
print("torch:", torch.__version__, "device:", device)

torch: 2.13.0+cpu device: cpu


## 1. 从监督 ASR 到自监督预训练

监督 ASR 需要 `(音频, 转录)`：

```text
音频 → Encoder → CTC/RNN-T/LLM → “今天天气很好”
```

自监督预训练只需要音频：

```text
音频 → 局部声学特征 → 随机遮挡 → Context Encoder
                              ↓
                    预测被遮挡位置的离散声学目标
```

- wav2vec 2.0：量化潜在表示作为目标，以对比学习区分正确目标与负样本；
- HuBERT：先聚类得到伪标签，再做 masked classification，并可迭代聚类；
- WavLM：在此基础上强调噪声、说话人和重叠语音等鲁棒性。

它们的共同点不是某个 API，而是：**把输入的一部分隐藏，迫使模型用上下文学习语音结构。**

## 2. 构造透明的合成“语音”

我们用 8 个离散声学单元模拟音素。每个单元有一个特征原型，同一个单元连续保持 3 帧，再叠加噪声。这保留了语音最重要的教学性质之一：相邻帧具有强相关性。

In [2]:
NUM_CODES = 8
FEAT_DIM = 16
SEQ_LEN = 48
SEGMENT = 3

generator = torch.Generator().manual_seed(123)
prototypes = F.normalize(
    torch.randn(NUM_CODES, FEAT_DIM, generator=generator), dim=-1
)


def make_synthetic_audio(batch_size, generator=None):
    generator = generator or torch.default_generator
    num_segments = math.ceil(SEQ_LEN / SEGMENT)
    segment_codes = torch.randint(
        0, NUM_CODES, (batch_size, num_segments), generator=generator
    )
    codes = segment_codes.repeat_interleave(SEGMENT, dim=1)[:, :SEQ_LEN]
    noise = 0.12 * torch.randn(
        batch_size, SEQ_LEN, FEAT_DIM, generator=generator
    )
    features = prototypes[codes] + noise
    return features, codes


features, codes = make_synthetic_audio(2, torch.Generator().manual_seed(1))
print("features:", tuple(features.shape), "targets:", tuple(codes.shape))
print("第一条离散单元:", codes[0, :18].tolist())
assert features.shape == (2, 48, 16)
assert torch.equal(codes[0, 0::3], codes[0, 1::3])
print("断言通过：离散单元连续 3 帧，特征带有噪声。")

features: (2, 48, 16) targets: (2, 48)
第一条离散单元: [5, 5, 5, 3, 3, 3, 4, 4, 4, 0, 0, 0, 7, 7, 7, 1, 1, 1]
断言通过：离散单元连续 3 帧，特征带有噪声。


## 3. Masked Acoustic Model

输入投影后，把选中的位置替换成同一个可学习 `mask_embedding`。Transformer 必须根据未遮挡的邻居恢复目标 code。

如果把原始目标帧继续交给模型，它只需识别该帧原型，不必学习上下文，这就是信息泄漏。

In [3]:
def make_mask(batch_size, seq_len, probability=0.35, generator=None):
    generator = generator or torch.default_generator
    mask = torch.rand(batch_size, seq_len, generator=generator) < probability
    # 保证每条样本至少有一个监督位置。
    for b in range(batch_size):
        if not mask[b].any():
            mask[b, 0] = True
    return mask


class MaskedAcousticModel(nn.Module):
    def __init__(self, feat_dim=FEAT_DIM, d_model=32, num_codes=NUM_CODES):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, d_model)
        self.mask_embedding = nn.Parameter(torch.zeros(d_model))
        self.position = nn.Parameter(torch.randn(1, SEQ_LEN, d_model) * 0.01)
        # 语音有很强的局部连续性。先加入局部卷积偏置，再由
        # Transformer 汇总更长上下文；这与 Conformer 的动机一致。
        self.local_context = nn.Conv1d(
            d_model, d_model, kernel_size=5, padding=2, groups=d_model
        )
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=96,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.context_encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.classifier = nn.Linear(d_model, num_codes)

    def encode(self, features, mask=None):
        x = self.input_proj(features)
        if mask is not None:
            replacement = self.mask_embedding.view(1, 1, -1)
            x = torch.where(mask.unsqueeze(-1), replacement, x)
        x = x + self.position[:, :x.size(1)]
        x = x + self.local_context(x.transpose(1, 2)).transpose(1, 2)
        return self.context_encoder(x)

    def forward(self, features, mask=None):
        return self.classifier(self.encode(features, mask))


ssl_model = MaskedAcousticModel().to(device)
mask = make_mask(2, SEQ_LEN, generator=torch.Generator().manual_seed(2))
ssl_logits = ssl_model(features, mask)
print("mask 比例:", mask.float().mean().item())
print("logits:", tuple(ssl_logits.shape))
assert ssl_logits.shape == (2, SEQ_LEN, NUM_CODES)
print("断言通过：每个时间步预测一个离散声学目标。")

mask 比例: 0.28125
logits: (2, 48, 8)
断言通过：每个时间步预测一个离散声学目标。


<USER_HOME>\AppData\Local\Temp\ipykernel_19412\4125103766.py:31: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.context_encoder = nn.TransformerEncoder(layer, num_layers=2)


## 4. 只在 mask 位置训练

损失为：

$$
\mathcal L=-\frac{1}{|M|}\sum_{t\in M}\log P(c_t\mid \tilde{x})
$$

$M$ 是被遮挡位置，$\tilde{x}$ 是遮挡后的输入。未遮挡帧提供上下文，但不计入本课的预测 loss。

In [4]:
def masked_loss_and_accuracy(logits, targets, mask):
    selected_logits = logits[mask]
    selected_targets = targets[mask]
    loss = F.cross_entropy(selected_logits, selected_targets)
    accuracy = (selected_logits.argmax(-1) == selected_targets).float().mean()
    return loss, accuracy


optimizer = torch.optim.AdamW(ssl_model.parameters(), lr=3e-3)
train_generator = torch.Generator().manual_seed(300)
history = []

ssl_model.train()
for step in range(121):
    batch_x, batch_codes = make_synthetic_audio(32, train_generator)
    batch_mask = make_mask(32, SEQ_LEN, 0.35, train_generator)
    logits = ssl_model(batch_x, batch_mask)
    loss, acc = masked_loss_and_accuracy(logits, batch_codes, batch_mask)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(ssl_model.parameters(), 5.0)
    optimizer.step()

    if step % 20 == 0:
        history.append((step, loss.item(), acc.item()))

for step, loss_value, acc_value in history:
    print(f"step={step:3d} loss={loss_value:.4f} masked_acc={acc_value:.3f}")

eval_generator = torch.Generator().manual_seed(999)
eval_x, eval_codes = make_synthetic_audio(128, eval_generator)
eval_mask = make_mask(128, SEQ_LEN, 0.35, eval_generator)
ssl_model.eval()
with torch.no_grad():
    eval_loss, eval_acc = masked_loss_and_accuracy(
        ssl_model(eval_x, eval_mask), eval_codes, eval_mask
    )

chance = 1 / NUM_CODES
print(f"评估 masked accuracy={eval_acc.item():.3f}; 随机猜测={chance:.3f}")
assert eval_acc > chance + 0.25
print("断言通过：模型确实从上下文恢复了大量被遮挡单元。")

step=  0 loss=2.2078 masked_acc=0.160
step= 20 loss=1.8207 masked_acc=0.297
step= 40 loss=1.3455 masked_acc=0.541
step= 60 loss=0.9573 masked_acc=0.698
step= 80 loss=0.9345 masked_acc=0.700
step=100 loss=0.7843 masked_acc=0.736
step=120 loss=0.4389 masked_acc=0.866
评估 masked accuracy=0.810; 随机猜测=0.125
断言通过：模型确实从上下文恢复了大量被遮挡单元。


### 你应该观察什么？

训练准确率不会必然单调，因为每一步的音频和 mask 都在变化。真正有意义的是固定随机种子的独立评估集，并与 `1/NUM_CODES` 的随机基线比较。

本例相邻 3 帧常属于同一个单元，所以模型可利用邻居。真实语音更复杂：目标可能来自量化器、K-means 聚类或教师模型，还需要处理说话人、噪声、音高和通道等变化。

## 5. 把预训练 encoder 接到 CTC

预训练阶段预测的是声学 code；ASR 微调阶段把 classifier 换成文字词表，并用 CTC/RNN-T/AED 等目标训练。Encoder 可以全部微调，也可以先冻结再逐步解冻。

In [5]:
class SSLCTCModel(nn.Module):
    def __init__(self, pretrained, vocab_size=10):
        super().__init__()
        self.pretrained = pretrained
        self.ctc_head = nn.Linear(32, vocab_size + 1)  # 0=blank

    def forward(self, x):
        hidden = self.pretrained.encode(x, mask=None)
        return self.ctc_head(hidden)


ctc_model = SSLCTCModel(ssl_model)
asr_x, _ = make_synthetic_audio(2, torch.Generator().manual_seed(44))
asr_targets = torch.tensor([1, 2, 3, 4, 2, 5, 6], dtype=torch.long)
input_lengths = torch.tensor([48, 48], dtype=torch.long)
target_lengths = torch.tensor([4, 3], dtype=torch.long)

ctc_logits = ctc_model(asr_x)
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)(
    ctc_logits.log_softmax(-1).transpose(0, 1),
    asr_targets,
    input_lengths,
    target_lengths,
)
ctc_model.zero_grad()
ctc_loss.backward()
encoder_grad = ctc_model.pretrained.input_proj.weight.grad.norm().item()
head_grad = ctc_model.ctc_head.weight.grad.norm().item()

print("CTC logits:", tuple(ctc_logits.shape))
print(f"loss={ctc_loss.item():.4f}, encoder_grad={encoder_grad:.4f}, head_grad={head_grad:.4f}")
assert ctc_logits.shape == (2, 48, 11)
assert encoder_grad > 0 and head_grad > 0
print("断言通过：预训练 encoder 与新的 CTC head 可以联合微调。")

CTC logits: (2, 48, 11)
loss=29.8548, encoder_grad=6.9625, head_grad=23.0195
断言通过：预训练 encoder 与新的 CTC head 可以联合微调。


## 6. 从教学实验到真实系统

真实预训练通常需要：

1. 数千到数百万小时经许可的多域音频；
2. 稳定的分布式训练、混合精度与 checkpoint；
3. 防止静音、重复、音乐、隐私数据和伪标签错误污染；
4. 下游 speaker-disjoint ASR 测试；
5. 比较“随机初始化”和“相同微调预算下的 SSL 初始化”，才可归因于预训练。

不要用“预训练 loss 下降”替代 CER/WER，也不要只在训练说话人上测试。

## 分层练习（24 分）

### A. 回忆（每题 1 分）

1. 自监督的“监督”来自哪里？
2. wav2vec 2.0 与 HuBERT 的目标构造有何不同？
3. 为什么要 mask 连续 span，而不只是单个点？
4. 预训练 classifier 为什么通常不能直接当文字输出层？

### B. 推理（每题 2 分）

5. `NUM_CODES=100` 时随机准确率是多少？
6. mask 比例为 0 会发生什么？比例为 1 又会怎样？
7. 为什么随机切分帧会造成 train/test 泄漏？
8. 何时应冻结 encoder，何时应全量微调？

### C. 编程（每题 3 分）

9. 把单点 mask 改为长度 3 的连续 span。
10. 比较带/不带位置编码的 masked accuracy。
11. 把 `SEGMENT` 从 3 改成 1，预测结果并验证。
12. 从空白实现 masked loss，保证只选择 `mask=True` 的位置。

达到 19/24，且能解释信息泄漏，才进入迷你音频语言模型。

## 离场小测（闭卷发给老师）

1. 为什么无文字音频仍能训练声学 encoder？
2. 本课为何只在 mask 位置算 loss？
3. 自监督预训练如何迁移到 CTC？
4. 写出至少两个会让 SSL 评估虚高的数据泄漏方式。

附上你的最终 masked accuracy 和最不确定的一题。